In [ ]:
import json
import os

from typing import List

import geopandas as gpd
import pandas as pd
import shapely.geometry as sg

from notebooks.dataset_utils import read_coco_annotations

RD_CRS = "EPSG:28992"  # CRS code for the Dutch Rijksdriehoek coordinate system
LAT_LON_CRS = "EPSG:4326"  # CRS code for WGS84 latitude/longitude coordinate system

In [ ]:
dataset_folder = "../datasets/oor/testride_velotech/"

images_folder = os.path.join(dataset_folder, "recording_2025-05-14_19-47-40/images")
coordinates_metadata_file = os.path.join(dataset_folder, "latest_track.json")
annotations_file = os.path.join(dataset_folder, "recording_2025-05-14_19-47-40_annotated.json")
cluster_annotations_file = os.path.join(dataset_folder, "recording_2025-05-14_reviewed.csv")

categories = {
    2: "Container",
    3: "Dixie",
    4: "Steiger",
}

In [ ]:
clusters_df = (
    pd.read_csv(cluster_annotations_file)
    .rename(columns={
        "image_name": "image_file_name",
        "ID": "cluster_id",
    })
)
clusters_df = clusters_df[clusters_df["object_category"].isin(categories.keys())]
clusters_df.sort_values(by="image_file_name", ascending=True, inplace=True)

In [ ]:
annotations_df = read_coco_annotations(json_file=annotations_file, categories=categories.keys())

In [ ]:
with open(coordinates_metadata_file, "r") as f:
    json_content = json.load(f)

data: dict[str, List] = {
    "image_file_name": [],
    "geometry": [],
}

for frame in json_content["frames"]:
    data["image_file_name"].append(frame["image_file_name"])
    data["geometry"].append(
        sg.Point(frame["gps_data"]["longitude"], frame["gps_data"]["latitude"])
    )

coordinates_metadata = gpd.GeoDataFrame(data=data, crs=LAT_LON_CRS).set_index("image_file_name")

In [ ]:
annotations_gdf = gpd.GeoDataFrame(
    data=annotations_df,
    geometry=annotations_df["image_file_name"].map(coordinates_metadata["geometry"])
)

In [ ]:
def get_cluster(row: pd.Series) -> List[int]:
    return clusters_df[
        (clusters_df["image_file_name"]==row["image_file_name"])
        & (clusters_df["object_category"]==row["category_id"])
    ]["cluster_id"].to_list()

annotations_clusters_merged = annotations_gdf.copy()
annotations_clusters_merged["cluster_ids"] = annotations_clusters_merged.apply(get_cluster, axis=1)

In [ ]:
# Optional: check duplicates

tmp = annotations_clusters_merged.reset_index().set_index(keys=["image_id", "category_id"])
tmp[tmp.index.duplicated(keep=False)].head(10)

In [ ]:
# Optional: write annotations with clusters to CSV file for manual corrections

clusters_check_file = os.path.join(dataset_folder, "clusters_checked.csv")
annotations_clusters_merged.drop(columns="geometry").to_csv(clusters_check_file)

In [ ]:
# Copy photos into folders per cluster to check

import cv2
from yolo_model_development_kit.inference_pipeline.source.output_image import OutputImage

clusters_output_folder = os.path.join(dataset_folder, "clusters_check")

def coco_box_to_bbox(yolo_box: List[float], img_width: int, img_height: int) -> List[int]:
    x_min, y_min, w, h = yolo_box
    x_max = x_min + w
    y_max = y_min + h

    x_min = int(x_min * img_width)
    x_max = int(x_max * img_width)
    y_min = int(y_min * img_height)
    y_max = int(y_max * img_height)

    return [x_min, y_min, x_max, y_max]

image_file_names = tmp["image_file_name"].unique()

for image_file in image_file_names:
    img_df = tmp[tmp["image_file_name"]==image_file]

    raw_image = cv2.imread(os.path.join(images_folder, image_file))
    img_width, img_height = raw_image.shape[1], raw_image.shape[0]

    # for cat_id in img_df["category_id"]:
    # cat_df = img_df[img_df["category_id"]==cat_id]
    cat_df = img_df
    bboxes = [
        coco_box_to_bbox(yolo_box, img_width, img_height)
        for yolo_box in cat_df["bbox"].to_list()
    ]
    obj_classes = cat_df["category_id"].to_list()
    names = cat_df.index.to_list()

    image = OutputImage(raw_image.copy())
    image.draw_bounding_boxes(
        boxes=bboxes, categories=obj_classes, tracking_ids=names
    )

    clusters = set().union(*[set(ids) for ids in cat_df["cluster_ids"].to_list()])
    # for cluster_id in clusters:
    # out_folder = os.path.join(clusters_folder, str(cluster_id))
    out_folder = os.path.join(clusters_output_folder, "ALL")
    os.makedirs(out_folder, exist_ok=True)
    cv2.imwrite(
        filename=os.path.join(out_folder, image_file),
        img=image.get_image()
    )